## README:

* Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually 

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ta

from dotenv import load_dotenv

In [2]:
# add project root to path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)

In [3]:
from data_processing.utils import get_indices_of_first_datapoint

## Load env variables

In [4]:
load_dotenv(r'../.env')

DATA_DIR = os.getenv('DATA_DIR')

## Global config

In [5]:
## ----- CONFIG ----- ##
WRITE_MODE = 'w' # 'x': do not overwrite

## Load data

In [6]:
data_df = pd.read_csv(os.path.join(DATA_DIR, 'preprocessed.csv'))
data_df.shape

(60840, 16)

In [7]:
data_df.columns

Index(['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'BAMLH0A0HYM2',
       'CPIAUCSL', 'DFF', 'ICSA', 'PPIACO', 'REAINTRATREARAT10Y', 'T10Y2Y',
       'USSTHPI', 'VIXCLS'],
      dtype='object')

In [8]:
data_df.isna().any().any()

np.False_

In [9]:
data_df['date'] = pd.to_datetime(data_df['date'], format='%Y-%m-%d')
data_df = data_df.set_index('date')

## Time features

In [10]:
time_df = data_df.copy()

In [11]:
## ----- CONFIG ----- ##
# base time features
time_df['month_of_year'] = time_df.index.month.values
time_df['day_of_week'] = time_df.index.day_of_week.values + 1 # Monday: 1, Sunday: 7

# Friday indicator (to capture "Friday Effect")
time_df['friday'] = (time_df['day_of_week'] == 5).astype(int)

# month sine/cosine
time_df['month_sin'] = np.sin(2 * np.pi * time_df['month_of_year'] / 12)
time_df['month_cos'] = np.cos(2 * np.pi * time_df['month_of_year'] / 12)

In [12]:
# summary of time features
print( time_df['month_of_year'].value_counts(sort=False) )
print( time_df['day_of_week'].value_counts(sort=False) )

month_of_year
1     4797
2     4662
3     5319
4     5022
5     5148
6     5157
7     5130
8     5391
9     4923
10    5364
11    4968
12    4959
Name: count, dtype: int64
day_of_week
5    12204
1    11412
2    12483
3    12483
4    12258
Name: count, dtype: int64


In [13]:
time_df.isna().any().any()

np.False_

## Technical indicators

In [14]:
## ----- CONFIG ----- ##
def add_indicators(grouped_data):
    """Add technical indicators to the stock prices.
    """
    # MACD
    macd = ta.trend.MACD(grouped_data['close'])
    grouped_data['macd'] = macd.macd()
    grouped_data['macd_signal'] = macd.macd_signal()
    grouped_data['macd_hist'] = macd.macd_diff()

    # RSI
    rsi = ta.momentum.RSIIndicator(
        grouped_data['close'], 
        window=30
    )
    grouped_data['rsi_30'] = rsi.rsi()

    # CCI
    cci = ta.trend.CCIIndicator(
        high=grouped_data['high'], 
        low=grouped_data['low'], 
        close=grouped_data['close'], 
        window=30
    )
    grouped_data['cci_30'] = cci.cci()

    return grouped_data

In [15]:
indicators_df = (
    time_df
    .groupby('tic')
    .apply(add_indicators, include_groups=False)
    .reset_index(level=0)
)
indicators_df.shape

(60840, 25)

In [16]:
na_df = indicators_df.isna()
na_perc = na_df.mean()
na_perc[na_perc > 0]

macd           0.003698
macd_signal    0.004882
macd_hist      0.004882
rsi_30         0.004290
cci_30         0.004290
dtype: float64

In [17]:
get_indices_of_first_datapoint(na_df)

,first datapoint,column
21,1999-03-11,macd_signal
22,1999-03-11,macd_hist
23,1999-03-05,rsi_30
24,1999-03-05,cci_30
20,1999-03-01,macd
5,1999-01-22,volume
0,1999-01-22,tic
4,1999-01-22,open
18,1999-01-22,month_sin
15,1999-01-22,month_of_year


In [18]:
## ----- CONFIG ----- ##
# choose a start date
START_DATE = '1999-03-11'

# choose feature to exclude
EXCLUDE_COLS = []

In [19]:
# subset columns
cols_filter = ~indicators_df.columns.isin(EXCLUDE_COLS)

# check NAs of subset
print('NAs detected:', na_df.loc[START_DATE:, cols_filter].any().any())
print('Data points per ticker:', indicators_df.loc[START_DATE:, cols_filter].shape[0])

NAs detected: False
Data points per ticker: 60543


## Finalize and export

In [20]:
export_df = indicators_df.reset_index()

print(export_df.shape)
export_df.columns

(60840, 26)


Index(['date', 'tic', 'close', 'high', 'low', 'open', 'volume', 'BAMLH0A0HYM2',
       'CPIAUCSL', 'DFF', 'ICSA', 'PPIACO', 'REAINTRATREARAT10Y', 'T10Y2Y',
       'USSTHPI', 'VIXCLS', 'month_of_year', 'day_of_week', 'friday',
       'month_sin', 'month_cos', 'macd', 'macd_signal', 'macd_hist', 'rsi_30',
       'cci_30'],
      dtype='object')

In [21]:
# export data
out_path = os.path.join(DATA_DIR, 'data_with_features.csv')
export_df.to_csv(out_path, index=False, mode=WRITE_MODE)